In [9]:
import pandas as pd
import matplotlib.pyplot as plt
import numpy as np

In [10]:
import os

DATA_DIR = os.getcwd()  # file .npy berada di direktori yang sama dengan notebook

# Load embeddings (shape: n_samples x 300)
questions_emb  = np.load(os.path.join(DATA_DIR, 'aug_questions_emb.npy'))
answerkeys_emb = np.load(os.path.join(DATA_DIR, 'aug_answerkeys_emb.npy'))
answers_emb    = np.load(os.path.join(DATA_DIR, 'aug_answers_emb.npy'))

# Metadata langsung dari pickle
meta_path = os.path.join(DATA_DIR, 'aug_metadata.pkl')
if os.path.exists(meta_path):
    metadata = pd.read_pickle(meta_path)
else:
    raise FileNotFoundError("aug_metadata.pkl tidak ditemukan di folder notebook.")

assert len(metadata) == len(answers_emb), \
    f"Mismatch: metadata={len(metadata)}, answers_emb={len(answers_emb)}"

print("=== Hasil Load ===")
print(f"questions_emb  : {questions_emb.shape}")
print(f"answerkeys_emb : {answerkeys_emb.shape}")
print(f"answers_emb    : {answers_emb.shape}")
print(f"\nMetadata       : {len(metadata)} rows")
print(f"Kolom metadata : {list(metadata.columns)}")
print(f"\nIDPSJ unik     : {sorted(metadata['IDPSJ'].unique())}")
print(f"\nDistribusi grade:")
print(metadata['grade'].value_counts().sort_index())


=== Hasil Load ===
questions_emb  : (12, 135, 768)
answerkeys_emb : (12, 90, 768)
answers_emb    : (1189, 80, 768)

Metadata       : 1189 rows
Kolom metadata : ['IDJwb', 'IDPSJ', 'grade', 'psj_idx']

IDPSJ unik     : [1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12]

Distribusi grade:
grade
1     122
2     151
3     112
4     112
5      65
6     115
7      88
8     101
9     100
10    223
Name: count, dtype: int64


In [11]:
# ── Analisis Distribusi per IDPSJ untuk Persiapan Mixup ──────────────────────

# Pivot: baris=IDPSJ, kolom=grade, nilai=jumlah sampel
pivot = (metadata.groupby(['IDPSJ', 'grade'])
                 .size()
                 .unstack(fill_value=0)
                 .reindex(columns=range(1, 11), fill_value=0))

print("=" * 70)
print("Jumlah sampel per grade per IDPSJ")
print("=" * 70)
print(pivot.to_string())

# Statistik ringkas per IDPSJ
stats = pd.DataFrame({
    'total'   : pivot.sum(axis=1),
    'max_cls' : pivot.max(axis=1),
    'min_cls' : pivot[pivot > 0].min(axis=1),
    'n_kelas' : (pivot > 0).sum(axis=1),
})
stats['median_cls'] = pivot.replace(0, np.nan).median(axis=1)
stats['target_1:2'] = (stats['max_cls'] / 2).apply(np.ceil).astype(int)
stats['target_1:3'] = (stats['max_cls'] / 3).apply(np.ceil).astype(int)

print("\n" + "=" * 70)
print("Statistik per IDPSJ")
print("=" * 70)
print(stats.to_string())

# Ringkasan global
global_max = pivot.max().max()
global_target_half = int(np.ceil(global_max / 2))
print(f"\nNilai kelas terbanyak secara global : {global_max}")
print(f"Target rasio 1:2 (max/2)            : {global_target_half}")
print(f"Target flat 10                      : 10")
print(f"\nKelas yang AKAN di-augmentasi (< target 1:2) per IDPSJ:")
for psj in pivot.index:
    row = pivot.loc[psj]
    local_max = row.max()
    tgt = int(np.ceil(local_max / 2))
    kurang = row[(row > 0) & (row < tgt)]
    if not kurang.empty:
        detail = ", ".join([f"grade {g}:{cnt}→{tgt}" for g, cnt in kurang.items()])
        print(f"  IDPSJ {psj} (target={tgt}): {detail}")


Jumlah sampel per grade per IDPSJ
grade  1   2   3   4   5   6   7   8   9   10
IDPSJ                                        
1       9   1  19   9   1  25   3  11   5  37
2       1   8  15   8   5   5   8  10   8   5
3       5   5   5   4   1  10   5   6   5   7
4       6   2   3  11   1   5   5   6   1   4
5      17   9   5   9   5   5   2   3   9   5
6      22  11  11  11   5   5   5  11  11   7
7       6   7   3   7   8   5   2   3   7  13
8      27  27  12   6  11   7   3  10  10  53
9      14  24   9   9   1  10  10   2   1  47
10      5   8  12  10  12  10  18  23  19  21
11      6  42  17  21  10  21  13  10  21  22
12      4   7   1   7   5   7  14   6   3   2

Statistik per IDPSJ
       total  max_cls  min_cls  n_kelas  median_cls  target_1:2  target_1:3
IDPSJ                                                                      
1        120       37        1       10         9.0          19          13
2         73       15        1       10         8.0           8          

In [12]:
# ── Mixup Augmentasi pada Embedding 300D ─────────────────────────────────────
# Strategi : intra-class Mixup per IDPSJ, hanya pada answers_emb
# questions_emb & answerkeys_emb TIDAK di-mix karena disimpan compact (per IDPSJ)
# Target   : ceil(max_cls_per_IDPSJ / 2) untuk setiap kelas minoritas
# lambda   : Beta(alpha=0.4) di-clip ke [0.5, 1.0] agar label tidak melompat

RNG = np.random.default_rng(seed=42)
ALPHA = 0.4

syn_answers = []
syn_meta = []

for psj in sorted(metadata['IDPSJ'].unique()):
    mask_psj = metadata['IDPSJ'] == psj
    psj_idx = metadata.index[mask_psj].to_numpy()
    psj_meta = metadata.loc[mask_psj].reset_index(drop=True)
    local_max = psj_meta['grade'].value_counts().max()
    target = int(np.ceil(local_max / 2))

    for grade in range(1, 11):
        cls_local_idx = psj_meta.index[psj_meta['grade'] == grade].to_numpy()
        n_cls = len(cls_local_idx)
        if n_cls == 0 or n_cls >= target:
            continue  # tidak perlu augmentasi

        n_needed = target - n_cls
        global_cls_idx = psj_idx[cls_local_idx]  # indeks ke answers_emb

        # Sampel pasangan secara acak (dengan penggantian)
        idx_i = RNG.choice(global_cls_idx, size=n_needed, replace=True)
        idx_j = RNG.choice(global_cls_idx, size=n_needed, replace=True)

        # lambda ~ Beta(alpha, alpha) di-clip ke [0.5, 1.0]
        lam = RNG.beta(ALPHA, ALPHA, size=n_needed)
        lam = np.clip(lam, 0.5, 1.0)

        # Reshape lambda agar bisa broadcast dengan dimensi embedding (n, ..., 300)
        lam_bc = lam.reshape((n_needed,) + (1,) * (answers_emb.ndim - 1))

        # Mixup hanya pada answers_emb
        syn_answers.append(lam_bc * answers_emb[idx_i] + (1 - lam_bc) * answers_emb[idx_j])

        # Label sintetis (intra-class -> grade tetap sama)
        syn_grade = np.full(n_needed, grade, dtype=int)
        for k in range(n_needed):
            syn_meta.append({
                'IDJwb': f'syn_{psj}_{grade}_{k}',
                'IDPSJ': psj,
                'grade': syn_grade[k],
            })

# ── Gabungkan data asli + sintetis ───────────────────────────────────────────
if syn_answers:
    aug_answers_emb = np.vstack([answers_emb] + syn_answers)
    aug_metadata = pd.concat(
        [metadata.drop(columns=['psj_idx'], errors='ignore'), pd.DataFrame(syn_meta)],
        ignore_index=True,
    )
else:
    aug_answers_emb = answers_emb.copy()
    aug_metadata = metadata.drop(columns=['psj_idx'], errors='ignore').copy()

# Tetap compact: 1 baris per IDPSJ
aug_questions_emb = questions_emb.copy()
aug_answerkeys_emb = answerkeys_emb.copy()

n_syn = len(aug_metadata) - len(metadata)
print(f"Data asli      : {len(metadata)}")
print(f"Data sintetis  : {n_syn}")
print(f"Total          : {len(aug_metadata)}")
print(f"\naug_answers_emb    : {aug_answers_emb.shape}")
print(f"aug_questions_emb  : {aug_questions_emb.shape} (compact per IDPSJ)")
print(f"aug_answerkeys_emb : {aug_answerkeys_emb.shape} (compact per IDPSJ)")
print(f"\nDistribusi grade setelah augmentasi:")
print(aug_metadata['grade'].value_counts().sort_index())

Data asli      : 1189
Data sintetis  : 554
Total          : 1743

aug_answers_emb    : (1743, 80, 768)
aug_questions_emb  : (12, 135, 768) (compact per IDPSJ)
aug_answerkeys_emb : (12, 90, 768) (compact per IDPSJ)

Distribusi grade setelah augmentasi:
grade
1     175
2     177
3     163
4     161
5     157
6     167
7     169
8     170
9     163
10    241
Name: count, dtype: int64


In [13]:
# ── Analisis Distribusi per IDPSJ untuk Persiapan Mixup ──────────────────────

# Pivot: baris=IDPSJ, kolom=grade, nilai=jumlah sampel
pivot = (aug_metadata.groupby(['IDPSJ', 'grade'])
                 .size()
                 .unstack(fill_value=0)
                 .reindex(columns=range(1, 11), fill_value=0))

print("=" * 70)
print("Jumlah sampel per grade per IDPSJ")
print("=" * 70)
print(pivot.to_string())

# Statistik ringkas per IDPSJ
stats = pd.DataFrame({
    'total'   : pivot.sum(axis=1),
    'max_cls' : pivot.max(axis=1),
    'min_cls' : pivot[pivot > 0].min(axis=1),
    'n_kelas' : (pivot > 0).sum(axis=1),
})
stats['median_cls'] = pivot.replace(0, np.nan).median(axis=1)
stats['target_1:2'] = (stats['max_cls'] / 2).apply(np.ceil).astype(int)
stats['target_1:3'] = (stats['max_cls'] / 3).apply(np.ceil).astype(int)

print("\n" + "=" * 70)
print("Statistik per IDPSJ")
print("=" * 70)
print(stats.to_string())

# Ringkasan global
global_max = pivot.max().max()
global_target_half = int(np.ceil(global_max / 2))
print(f"\nNilai kelas terbanyak secara global : {global_max}")
print(f"Target rasio 1:2 (max/2)            : {global_target_half}")
print(f"Target flat 10                      : 10")
print(f"\nKelas yang AKAN di-augmentasi (< target 1:2) per IDPSJ:")
for psj in pivot.index:
    row = pivot.loc[psj]
    local_max = row.max()
    tgt = int(np.ceil(local_max / 2))
    kurang = row[(row > 0) & (row < tgt)]
    if not kurang.empty:
        detail = ", ".join([f"grade {g}:{cnt}→{tgt}" for g, cnt in kurang.items()])
        print(f"  IDPSJ {psj} (target={tgt}): {detail}")


Jumlah sampel per grade per IDPSJ
grade  1   2   3   4   5   6   7   8   9   10
IDPSJ                                        
1      19  19  19  19  19  25  19  19  19  37
2       8   8  15   8   8   8   8  10   8   8
3       5   5   5   5   5  10   5   6   5   7
4       6   6   6  11   6   6   6   6   6   6
5      17   9   9   9   9   9   9   9   9   9
6      22  11  11  11  11  11  11  11  11  11
7       7   7   7   7   8   7   7   7   7  13
8      27  27  27  27  27  27  27  27  27  53
9      24  24  24  24  24  24  24  24  24  47
10     12  12  12  12  12  12  18  23  19  21
11     21  42  21  21  21  21  21  21  21  22
12      7   7   7   7   7   7  14   7   7   7

Statistik per IDPSJ
       total  max_cls  min_cls  n_kelas  median_cls  target_1:2  target_1:3
IDPSJ                                                                      
1        214       37       19       10        19.0          19          13
2         89       15        8       10         8.0           8          

In [14]:
# ── Simpan hasil akhir ───────────────────────────────────────────────────────
# answers_emb : simpan penuh karena setiap sampel unik
# questions & answerkeys : simpan HANYA 1 per IDPSJ (hemat storage)
#   -> rekonstruksi di training menggunakan kolom psj_idx di metadata

import pickle

# Buat mapping IDPSJ -> indeks unik
idpsj_sorted = sorted(aug_metadata['IDPSJ'].unique())
idpsj_to_idx = {psj: i for i, psj in enumerate(idpsj_sorted)}
final_metadata = aug_metadata.copy()
final_metadata['psj_idx'] = final_metadata['IDPSJ'].map(idpsj_to_idx)

# Handle dua kemungkinan format input:
# 1) compact (n_idpsj, seq_len, 300)
# 2) expanded (n_samples, seq_len, 300) -> dipadatkan lagi
if aug_questions_emb.shape[0] == len(idpsj_sorted):
    uniq_questions_emb = aug_questions_emb
elif aug_questions_emb.shape[0] == len(final_metadata):
    uniq_q_rows = [final_metadata.index[final_metadata['IDPSJ'] == psj][0] for psj in idpsj_sorted]
    uniq_questions_emb = aug_questions_emb[uniq_q_rows]
else:
    raise ValueError(
        f"Ukuran aug_questions_emb tidak cocok: {aug_questions_emb.shape[0]} "
        f"(expected {len(idpsj_sorted)} atau {len(final_metadata)})"
    )

if aug_answerkeys_emb.shape[0] == len(idpsj_sorted):
    uniq_answerkeys_emb = aug_answerkeys_emb
elif aug_answerkeys_emb.shape[0] == len(final_metadata):
    uniq_q_rows = [final_metadata.index[final_metadata['IDPSJ'] == psj][0] for psj in idpsj_sorted]
    uniq_answerkeys_emb = aug_answerkeys_emb[uniq_q_rows]
else:
    raise ValueError(
        f"Ukuran aug_answerkeys_emb tidak cocok: {aug_answerkeys_emb.shape[0]} "
        f"(expected {len(idpsj_sorted)} atau {len(final_metadata)})"
    )

# Simpan
np.save(os.path.join(DATA_DIR, 'final_answers_emb.npy'), aug_answers_emb)
np.save(os.path.join(DATA_DIR, 'final_questions_emb.npy'), uniq_questions_emb)
np.save(os.path.join(DATA_DIR, 'final_answerkeys_emb.npy'), uniq_answerkeys_emb)

# Simpan metadata dengan pickle protocol 4 (lebih kompatibel lintas environment)
with open(os.path.join(DATA_DIR, 'final_metadata.pkl'), 'wb') as f:
    pickle.dump(final_metadata, f, protocol=4)

print("File tersimpan:")
print(f"  final_answers_emb.npy    {aug_answers_emb.shape}   (per sampel)")
print(f"  final_questions_emb.npy  {uniq_questions_emb.shape}  (per IDPSJ unik)")
print(f"  final_answerkeys_emb.npy {uniq_answerkeys_emb.shape} (per IDPSJ unik)")
print(f"  final_metadata.pkl       {len(final_metadata)} rows  (+ kolom psj_idx)")

before_mb = (aug_questions_emb.nbytes + aug_answerkeys_emb.nbytes) / 1024**2
after_mb = (uniq_questions_emb.nbytes + uniq_answerkeys_emb.nbytes) / 1024**2
print(f"\nHemat storage questions+answerkeys: {before_mb:.1f} MB -> {after_mb:.1f} MB")

File tersimpan:
  final_answers_emb.npy    (1743, 80, 768)   (per sampel)
  final_questions_emb.npy  (12, 135, 768)  (per IDPSJ unik)
  final_answerkeys_emb.npy (12, 90, 768) (per IDPSJ unik)
  final_metadata.pkl       1743 rows  (+ kolom psj_idx)

Hemat storage questions+answerkeys: 15.8 MB -> 15.8 MB
